## 1. Setup — Drive e clone della repo MSAE

Monta Google Drive e clona la repo ufficiale [WolodjaZ/MSAE](https://github.com/WolodjaZ/MSAE).




## 0. Configurazione — Expansion factor

Scegli qui se addestrare il MSAE con `expansion_factor = 8` (latente 6144, come nel paper) o `16` (latente 12288). Questa è l'unica cella da toccare: la dimensione latente, il `max_nesting` per la K-list Matryoshka e il flag `--expansion_factor` passato a `train.py` vengono derivati automaticamente, coerentemente con la convenzione usata negli altri notebook della pipeline.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CONFIGURAZIONE — Expansion factor del MSAE
# ════════════════════════════════════════════════════════════════════
# Unico punto da modificare per scegliere la capacita' latente del SAE:
#   8  -> latente 6144  (come nel paper di riferimento)
#   16 -> latente 12288 (capacita' raddoppiata)
# Le celle successive (patch di config.py, K-list Matryoshka, lancio di
# train.py) si adattano automaticamente a questo valore.
EXPANSION_FACTOR = 16          # <-- cambia qui: 8 oppure 16

assert EXPANSION_FACTOR in (8, 16), "EXPANSION_FACTOR deve essere 8 o 16"

CLIP_DIM   = 768
LATENT_DIM = CLIP_DIM * EXPANSION_FACTOR   # = max_nesting (copre tutta la latente)

# K-list Matryoshka: la repo raddoppia da nesting_list=64 fino a max_nesting.
# La ricostruiamo qui solo per documentazione/sanity check.
NESTING_START = 64
K_LIST = []
_k = NESTING_START
while _k < LATENT_DIM:
    K_LIST.append(_k)
    _k *= 2
K_LIST.append(LATENT_DIM)

print(f"Expansion factor  : {EXPANSION_FACTOR}")
print(f"Dimensione latente: {LATENT_DIM}")
print(f"max_nesting       : {LATENT_DIM}")
print(f"K-list Matryoshka : {K_LIST}  ({len(K_LIST)} livelli)")

In [ ]:
from google.colab import drive
import pathlib

drive.mount('/content/drive')

# Clone della repo MSAE
!git clone https://github.com/WolodjaZ/MSAE.git /content/MSAE

%cd /content/MSAE


In [ ]:
import shutil
from pathlib import Path

PATCHES = Path("/content/drive/MyDrive/tesi/msae_patches")

for fname in ["train.py", "utils.py"]:
    src = PATCHES / fname
    dst = Path("/content/MSAE") / fname
    if not src.exists():
        raise FileNotFoundError(
            f"File non trovato su Drive: {src}\n"
            f"Carica MSAE/{fname} in MyDrive/tesi/msae_patches/ prima di procedere."
        )
    shutil.copy(src, dst)
    print(f"Sostituito: {fname}")

## 1ter. Sostituzione dei file MSAE modificati

Il `git clone` porta la repo originale. Sovrascriviamo `train.py` e `utils.py` con le versioni modificate che aggiungono:
- early stopping con patience=20 (`train.py`)
- argomento `-dv / --dataset_val` (`train.py`)
- supporto a parametri di normalizzazione esterni in `SAEDataset` (`utils.py`)

**Prerequisito**: caricare `train.py` e `utils.py` patchati in `MyDrive/tesi/msae_patches/`.

## 1bis. Patch dei default della repo MSAE

Due default in `config.py` non corrispondono al setup del paper sul nostro subset:

- **`max_nesting=256`** → la repo costruisce la K-list Matryoshka raddoppiando da `nesting_list=64` fino a `max_nesting`. Serve una K-list che copra tutta la latente, quindi impostiamo `max_nesting = LATENT_DIM` (= 768 × `EXPANSION_FACTOR`, definito nella cella 0):
  - con `expansion_factor=8` → latente 6144 → `[64, 128, 256, 512, 1024, 2048, 4096, 6144]` (8 livelli)
  - con `expansion_factor=16` → latente 12288 → `[64, 128, 256, 512, 1024, 2048, 4096, 8192, 12288]` (9 livelli)

- **`batch_size=4096`** → su 26 325 sample del subset darebbe solo ~6 step di gradiente per epoca. Portandolo a `256` otteniamo ~103 step/epoca → ~10k step totali su 100 epoche, ordine di grandezza dei ~21k step del paper su CC3M (708 step/epoca × 30 epoche).

`train.py` non espone questi due parametri come CLI flag, quindi li patchiamo direttamente in `config.py` prima del lancio. `--expansion_factor` ed `--epochs` sono invece esposti e vengono passati dalla cella di training (rispettivamente dalla variabile `EXPANSION_FACTOR` e con default 100, qui 200).

In [ ]:
import re
import subprocess
from pathlib import Path

cfg_path = Path("/content/MSAE/config.py")
src = cfg_path.read_text()

# max_nesting: 256 -> LATENT_DIM (K-list completa fino a coprire tutta la latente)
# LATENT_DIM = 768 * EXPANSION_FACTOR (6144 se 8, 12288 se 16), definito nella cella 0.
src_new = re.sub(#con una regex cerco dove il max nesting è impostato a 256 e lo imposto a LATENT_DIM
    r"max_nesting:\s*int\s*=\s*256[^\n]*",
    f"max_nesting: int = {LATENT_DIM}",
    src,
)
# batch_size: 4096 -> 256 (piu' step di gradiente su dataset piccolo)
src_new = re.sub(#faccio la stessa cosa per modificare la batch size
    r"batch_size:\s*int\s*=\s*4096[^\n]*",
    "batch_size: int = 256",
    src_new,
)

assert src_new != src, "Sostituzioni fallite: controlla manualmente config.py"
cfg_path.write_text(src_new)

# Verifica
out = subprocess.run(
    ["grep", "-nE", "(max_nesting|batch_size):", str(cfg_path)],
    capture_output=True, text=True,
).stdout
print(f"max_nesting impostato a {LATENT_DIM} (expansion_factor={EXPANSION_FACTOR})")
print("Linee patchate in config.py:")
print(out)

## 2. Copia degli embedding precomputati da Drive

I tre file `.npy` sono raw memmap prodotti dal notebook `6-embedding_extraction.ipynb` con split stratificato 70/20/10 per cluster:
- `places365_ViT-L~14_train_image_20475_768.npy` — 70% del subset (training)
- `places365_ViT-L~14_val_image_5850_768.npy` — 20% del subset (validation, usato per early stopping)
- `places365_ViT-L~14_test_image_2925_768.npy` — 10% del subset (test, valutato una sola volta a fine training)

Vengono copiati in `MSAE/data/` per matchare la convenzione di path della repo.

In [ ]:
import shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/tesi")
DATA  = Path("/content/MSAE/data")
DATA.mkdir(exist_ok=True)

# Cerca i file tramite glob: le dimensioni esatte dipendono dallo split stratificato
patterns = [
    "places365_ViT-L~14_train_image_*_768.npy",
    "places365_ViT-L~14_val_image_*_768.npy",
    "places365_ViT-L~14_test_image_*_768.npy",
]

for pattern in patterns:
    matches = list(DRIVE.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Nessun file corrispondente a '{pattern}' in {DRIVE}")
    src = matches[0]
    dst = DATA / src.name
    shutil.copy(src, dst)
    print(f"Copiato: {src.name}  ({dst.stat().st_size / 1e6:.1f} MB)")

## 4. Training del MSAE

Lancia `train.py` con la configurazione **MSAE_RW** (Matryoshka SAE con reverse weighting):

- `-dt` / `-dv` / `-ds`: file di train, validation e test in formato raw memmap
- `-m MSAE_RW`: variante Matryoshka con pesatura inversa `{h, h-1, ..., 1}` , secondo il paper RW migliora la sparsity (L0 ~0.83 vs 0.75 di UW) con degrado trascurabile della reconstruction quality
- `-a TopKReLU`: attivazione che enforce sparsity tramite top-k selection (TopK rimosso a inference, resta solo ReLU)
- `--expansion_factor {EXPANSION_FACTOR}`: passato dalla variabile della cella 0 → latente 768 × `EXPANSION_FACTOR` (8 → 6144 come nel paper; 16 → 12288, capacità raddoppiata)
- `--epochs 200`: tetto massimo; l'early stopping (patience=20 epoche senza miglioramento di val_loss) interromperà prima se necessario. Il modello salvato corrisponde sempre al best checkpoint sulla validation.

Dopo il patch di `config.py` la K-list Matryoshka coincide con `K_LIST` calcolata nella cella 0 (8 livelli con expansion 8, 9 livelli con expansion 16).

**Cosa monitorare nei log delle prime 5 epoche** (sanity check per decidere se proseguire):
- **CKNNA All / CKNNA Sparse** → devono **salire** (target paper ~0.99). Se scendono o restano piatte, il dataset è troppo piccolo per allenare l'intera latente: accetta il limite, riduci l'expansion factor o amplia il subset.
- **FVU Sparse / FVU All** → devono scendere (target paper < 0.1)
- **Sparsity (L0)** → target ~0.8 (80% dei latenti a zero)
- **Dead neurons** → idealmente vicino a zero

Il training salva un file `.pth` nella cwd a fine training.

In [ ]:
import glob

DRIVE_TESI = "/content/drive/MyDrive/tesi"

train_f = glob.glob(f"{DRIVE_TESI}/places365_ViT-L~14_train_image_*_768.npy")[0]
val_f   = glob.glob(f"{DRIVE_TESI}/places365_ViT-L~14_val_image_*_768.npy")[0]
test_f  = glob.glob(f"{DRIVE_TESI}/places365_ViT-L~14_test_image_*_768.npy")[0]

print(f"Train : {train_f}")
print(f"Val   : {val_f}")
print(f"Test  : {test_f}")
print(f"Expansion factor: {EXPANSION_FACTOR}  ->  latente {LATENT_DIM}")

!python train.py \
  -dt {train_f} \
  -dv {val_f} \
  -ds {test_f} \
  -m MSAE_RW -a TopKReLU \
  --expansion_factor {EXPANSION_FACTOR} --epochs 200

## 5. Salvataggio del modello su Drive

Il file `.pth` viene generato in `/content/MSAE/` con un nome auto-generato che codifica gli iperparametri: il primo campo è la dimensione latente, quindi cambia con l'expansion factor (es. `12288_768_TopKReLU_64_RW_..._train_image_20475_768.pth` con expansion 16, `6144_768_...` con expansion 8).

Lo copio su Drive così da poterlo riusare in fasi successive (estrazione SAE embedding, evaluation, fuzzy inference).

In [ ]:
import glob
import shutil
import pathlib

DRIVE = pathlib.Path("/content/drive/MyDrive/tesi")
MODELS_DIR = DRIVE / "models"
MODELS_DIR.mkdir(exist_ok=True)

pth_files = glob.glob("/content/MSAE/*.pth")
if not pth_files:
    raise FileNotFoundError("Nessun file .pth trovato in /content/MSAE/. Il training è andato a buon fine?")

for f in pth_files:
    src = pathlib.Path(f)
    dst = MODELS_DIR / src.name
    shutil.copy(src, dst)
    print(f"Salvato su Drive: {dst}  ({dst.stat().st_size / 1e6:.1f} MB)")